(rd.tracking)=
# 10. Introduction to Long Term Tracking

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

Up to this point, tracking has implicitly been limited to tracking once through the lattice, which
`twiss` does to compute the closed orbit and the lattice functions. SciBmad's `track` tracks any
number of particles over any number of turns, with spin, radiation, intra-beam scattering, and
lattice parameters that change with time (see [Track](../track.md)). The tracking is parallelized
on the CPU, and can run on GPUs.

## Initializing a Beam

The initial particles are given to `track` as a matrix with one row per particle and the six phase
space coordinates $x, p_x, y, p_y, z, p_z$ as its columns. Here are seven electrons with various
$z$ and $p_z$:

In [2]:
beam = np.array([[0, 0, 0, 0, 0,     0],
                 [0, 0, 0, 0, 0.1,   0],
                 [0, 0, 0, 0, 0.15,  0],
                 [0, 0, 0, 0, 0,    -0.01],
                 [0, 0, 0, 0, 0,     0.015],
                 [0, 0, 0, 0, 0,     0.02],
                 [0, 0, 0, 0, 0,     0.03]])

Such a matrix can also be read from a file, for example a text file with one row per particle with
{jl}`readdlm`{jltext}` of the `{jl}`DelimitedFiles`{jltext}` package`{py}`np.loadtxt`.

## Tracking

We track these electrons through the ring with the RF cavities of [chapter 9](rf.md), with
radiation damping in every element:

In [3]:
from lattices.ch09_rf import *
for ele in ring.line:
    ele.rad_damping = True

With radiation damping, the electrons lose energy in every bend, which the RF cavities replace.
The closed orbit is then not zero: it has a small $p_z$, which changes along the ring (the energy
is lost in the bends and gained in the cavities), and a $z$ such that the cavities restore the
energy lost in one turn. The particles above are relative to the closed orbit:

In [4]:
co = sb.find_closed_orbit(ring).v0  # 1×6 array
v0 = co + beam
res = sb.track(ring, v0=v0, n_turns=30)
res

TrackResult of 7 particles:
config:
 n_turns                         = 30
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 0.140203047 seconds

start = 1
turns = 0:1:30

s, index, name, kind    the saved locations: the end of the beamline (s = 3834.04)
state                   particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz     the coordinates, e.g. x(turn=n, particle=i)
v                       all six coordinates, e.g. v(turn=n, particle=i)
data                    the internal, low-le

The tracking saves the particles at the end of the ring after every turn. The state of each particle
after the last turn is in `res.state`: {jl}`0x1`{py}`1` is alive, and other values are the ways a
particle
can be lost (see [Track](../track.md)):

In [5]:
res.state(turn=30)

array([1, 1, 2, 1, 2, 2, 2], dtype=uint8)

The particles with the largest $z$ or $p_z$ are lost: they started outside of the RF bucket, the
region of stable longitudinal motion. The trajectories in the $z$-$p_z$ plane:

In [6]:
from matplotlib.figure import Figure
fig = Figure(figsize=(7, 4.5), layout="constrained")
ax = fig.subplots()
for i in range(1, 8):
    ax.scatter(res.z(turn=..., particle=i), res.pz(turn=..., particle=i), s=10,
               label=f"particle {i}")
ax.set(xlabel="z [m]", ylabel="pz", xlim=(-0.3, 0.3), ylim=(-0.04, 0.04))
ax.legend(loc="lower right")
fig

<Figure size 700x450 with 1 Axes>

{jl}`res.z[turn=:, particle=i]`{py}`res.z(turn=..., particle=i)` is $z$ of particle `i` after every
turn, from turn 0 (the initial
coordinates) to turn 30. The coordinates of a lost particle stay at the values it had where it was
lost (here outside of the plot). The RF wavelength, the length of the ring divided by the harmonic
number, is 0.51 m.

## Emittance and Radiation

The emittance of a beam is the average action $\langle J\rangle$ of its particles in each normal
mode. With radiation, the emittances approach equilibrium values, at which the radiation damping and
the random excitation by the emission of photons (the radiation fluctuations) balance. To see this
in a reasonable time, here is a small ring of 10 sectors, for positrons at 13 GeV:

In [7]:
with sb.elements():
    q1 = sb.Quadrupole(L=0.45, Kn1=1.0, tilt=0.01)  # Tilted to couple the modes
    q2 = sb.Quadrupole(L=0.55, Kn1=-1.0)
    d = sb.Drift(L=0.3)
    bs = sb.SBend(L=0.5, angle=0.01)
    rf = sb.RFCavity(L=0.1, rf_frequency=500e6, voltage=1e8,
                     zero_phase=sb.PhaseRef.AboveTransition)
simple_ring = sb.Beamline([d, q1, d, bs, d, q2, d, rf, bs]*10,
                          species_ref=sb.Species("positron"), pc_ref=13e9)
for ele in simple_ring.line:
    ele.rad_damping = True
    ele.rad_fluctuations = True
tws = sb.twiss(simple_ring, cols=["a"], at=[1])
tws

Twiss:
summ:
 q1     = 0.734460564953469
 q2     = 1.5301484348056853
 q3     = -0.03657303038671941
 etac   = 0.001972565289404409
 alphac = 0.001972566834492822
 damp1  = 6.130352746160272e-5
 damp2  = 6.186418181371888e-5
 damp3  = 0.00012428974677165054

df:
1×5 DataFrame
 Row │ index  name  kind   s    a
     │                     [m]
─────┼────────────────────────────────────────────────────────────
   1 │     1  d     Drift  0.0  [2.94007 0.0 … -0.00149422 0.655…

The summary values `damp1`, `damp2`, `damp3` are the damping decrements per turn: the amplitudes of
the modes decrease by the factor $e^{-\mathrm{damp}}$ per turn, and the emittances by
$e^{-2\,\mathrm{damp}}$. The column `a` is the matrix $A$ of the normalizing map at the start of the
ring: it takes the normalized coordinates $\tilde{x}$, in which the motion of each mode is a
(damped) rotation, to the phase space coordinates relative to the closed orbit. In the normalized
coordinates, the action of mode 1 is $J_1 = (\tilde{x}_1^2 + \tilde{x}_2^2)/2$, and likewise for the
other modes.

A beam of 100 particles is started with smaller emittances than the equilibrium ones, by drawing the
normalized coordinates from Gaussian distributions:

In [8]:
rng = np.random.default_rng(1)
A = tws.a[1]
co_s = sb.find_closed_orbit(simple_ring).v0
n = 100
emit0 = np.array([1e-9, 1e-12, 1e-8])  # Initial emittances of the three modes [m]
xn = rng.standard_normal((n, 6)) * np.sqrt(np.repeat(emit0, 2))
v0 = co_s + xn @ A.T
res = sb.track(simple_ring, v0=v0, n_turns=20000, save_every_n_turns=100)
res

TrackResult of 100 particles:
config:
 n_turns                         = 20000
 save_every_n_turns              = 100
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 27.657512587 seconds

start = 1
turns = 0:100:20000

s, index, name, kind       the saved locations: the end of the beamline (s = 33.0)
state                      particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz        the coordinates, e.g. x(turn=n, particle=i)
v                          all six coordinates, e.g. v(turn=n, particle=i)
data                  

The emittance of mode 1 after each saved turn:

In [9]:
Ai = np.linalg.inv(A)
emit1 = [np.mean(np.sum((((res.v(turn=t) - co_s) @ Ai.T)[:, :2])**2, axis=1))/2
         for t in res.turns]
fig = Figure(figsize=(7, 3.5), layout="constrained")
ax = fig.subplots()
ax.plot(res.turns, emit1)
ax.set(xlabel="turn", ylabel="ε₁ [m]")
fig

<Figure size 700x350 with 1 Axes>

The emittance approaches its equilibrium value with the damping time: after $n$ turns, the
difference from the equilibrium has decreased by the factor $e^{-2 n\,\mathrm{damp1}}$.

## Exercises

1. **Initial parameters:** Experiment with different initial particle coordinates and plot their
   trajectories. How much can you change $p_z$ without losing the particle? Track particles and plot
   their longitudinal phase space. Find the stable and unstable fixed points, and the invariant
   curves in the longitudinal phase space. (Turn radiation damping off to see invariant curves.)

2. **Equilibrium emittance:** Fit the emittance of the small ring with
   $\varepsilon(n) = \varepsilon_\infty + (\varepsilon_0 - \varepsilon_\infty) e^{-2n\,\mathrm{damp1}}$ to
   find the equilibrium emittance $\varepsilon_\infty$. (Bmad computes it from the radiation
   integrals: $7.12\cdot10^{-8}$ m.) How large is the statistical error with 100 particles?

3. **Emittance of mode 2:** Compute the emittance of mode 2 as well. What makes it nonzero? Hint:
   look at the definition of the quadrupole `q1`.